In [1]:
import os
print("Files in current directory:", os.listdir('.'))
for root, dirs, files in os.walk('.'):
    print(root, len(dirs), "dirs,", len(files), "files")
    if len(files) < 10:
        print("  Files:", files)

Files in current directory: ['.config', 'xnzhj3x8v4-1.zip', 'archive.zip', 'sample_data']
. 2 dirs, 2 files
  Files: ['xnzhj3x8v4-1.zip', 'archive.zip']
./.config 2 dirs, 8 files
  Files: ['config_sentinel', '.last_update_check.json', 'active_config', 'hidden_gcloud_config_universe_descriptor_data_cache_configs.db', '.last_opt_in_prompt.yaml', '.last_survey_prompt.yaml', 'gce', 'default_configs.db']
./.config/configurations 0 dirs, 1 files
  Files: ['config_default']
./.config/logs 1 dirs, 0 files
  Files: []
./.config/logs/2026.10.02 0 dirs, 6 files
  Files: ['13.50.21.555054.log', '13.50.08.874802.log', '13.50.15.366402.log', '13.50.14.261149.log', '13.49.46.128557.log', '13.50.21.004336.log']
./sample_data 0 dirs, 6 files
  Files: ['README.md', 'anscombe.json', 'california_housing_test.csv', 'mnist_test.csv', 'mnist_train_small.csv', 'california_housing_train.csv']


In [2]:
import zipfile
import os

with zipfile.ZipFile('xnzhj3x8v4-1.zip', 'r') as z:
    print("xnzhj3x8v4-1.zip top entries:", z.namelist()[:15])
    print("Total entries in xnzhj3x8v4-1.zip:", len(z.namelist()))
    z.extractall('asphalt_data')

with zipfile.ZipFile('archive.zip', 'r') as z:
    print("archive.zip entries:", z.namelist())
    z.extractall('email_data')

for root, dirs, files in os.walk('asphalt_data'):
    print(root, "->", len(dirs), "dirs,", len(files), "files")
for root, dirs, files in os.walk('email_data'):
    print(root, "->", len(dirs), "dirs,", len(files), "files")

xnzhj3x8v4-1.zip top entries: ['448/', '448/NonCracks/', '448/NonCracks/IMG_20180705_130741989 resized_448.jpg', '448/NonCracks/IMG_8710 resized_448.jpg', '448/NonCracks/IMG_8789 resized_448.jpg', '448/NonCracks/IMG_20180705_130743987 resized_448.jpg', '448/NonCracks/IMG_8800 resized_448.jpg', '448/NonCracks/IMG_8798 resized_448.jpg', '448/NonCracks/IMG_20180705_130742988 resized_448.jpg', '448/NonCracks/IMG_8701 resized_448.jpg', '448/NonCracks/IMG_8699 resized_448.jpg', '448/NonCracks/IMG_20180705_130748649 resized_448.jpg', '448/NonCracks/IMG_8673 resized_448.jpg', '448/NonCracks/IMG_8754 resized_448.jpg', '448/NonCracks/IMG_20180705_130737652 resized_448.jpg']
Total entries in xnzhj3x8v4-1.zip: 403
archive.zip entries: ['emails.csv']
asphalt_data -> 1 dirs, 0 files
asphalt_data/448 -> 2 dirs, 0 files
asphalt_data/448/Cracks -> 0 dirs, 200 files
asphalt_data/448/NonCracks -> 0 dirs, 200 files
email_data -> 0 dirs, 1 files


In [6]:
import zipfile
import os

# Extract xnzhj3x8v4-1.zip directly to current working directory ('.')
print("Extracting asphalt dataset...")
with zipfile.ZipFile('xnzhj3x8v4-1.zip', 'r') as z:
    z.extractall('.')

# Extract archive.zip directly to current working directory ('.')
print("Extracting email dataset...")
with zipfile.ZipFile('archive.zip', 'r') as z:
    z.extractall('.')

# Verify that the '448' folder and 'emails.csv' exist in the root folder
print("Root contents:", os.listdir('.'))
if os.path.exists('448'):
    print("Success! '448' directory is now located in the root workspace.")
else:
    print("Error: '448' directory could not be found.")

Extracting asphalt dataset...
Extracting email dataset...
Root contents: ['.config', 'emails.csv', '448', 'asphalt_data', 'xnzhj3x8v4-1.zip', 'email_data', 'archive.zip', 'sample_data']
Success! '448' directory is now located in the root workspace.


In [9]:
import os
import time
import numpy as np
import pandas as pd
import cv2
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# PART A: Image Feature Extraction & Classification
# ==========================================

def extract_image_features(image_path, target_size=(128, 128)):
    """Reads an image, resizes it, converts to grayscale, and extracts intensity and edge features."""
    img = cv2.imread(image_path)
    if img is None:
        return None

    img_resized = cv2.resize(img, target_size)
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

    mean_brightness = np.mean(gray)
    contrast = np.std(gray)

    dark_pixel_ratio = np.sum(gray < 50) / gray.size
    bright_pixel_ratio = np.sum(gray > 200) / gray.size

    edges = cv2.Canny(gray, threshold1=100, threshold2=200)
    edge_count = np.count_nonzero(edges)
    edge_density = edge_count / edges.size

    return [
        mean_brightness, contrast, dark_pixel_ratio,
        bright_pixel_ratio, edge_count, edge_density
    ]

def load_asphalt_dataset(base_dir="448"):
    """Walks through the asphalt dataset directories to extract features and labels."""
    data = []
    labels = []

    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, file)
                if "NonCrack" in root:
                    label = 0
                else:
                    label = 1 # Crack

                features = extract_image_features(img_path)
                if features is not None:
                    data.append(features)
                    labels.append(label)

    feature_columns = [
        "mean_brightness", "contrast", "dark_pixel_ratio",
        "bright_pixel_ratio", "edge_count", "edge_density"
    ]
    df_features = pd.DataFrame(data, columns=feature_columns)
    df_features['label'] = labels
    return df_features

print("--- Processing Part A: Images ---")
df_images = load_asphalt_dataset("448")
print(f"Extracted image feature shape: {df_images.shape}")

# ==========================================
# PART B: Pre-vectorized Email Data & Spam Classification
# ==========================================

print("\n--- Processing Part B: Pre-vectorized Email Data ---")
email_csv_path = "emails.csv"
if os.path.exists(email_csv_path):
    df_emails = pd.read_csv(email_csv_path)
    print("Email dataset shape:", df_emails.shape)

    label_col = 'Prediction'
    feature_cols = [col for col in df_emails.columns if col not in ['Email No.', label_col]]

    print("\nClass Distribution (Spam vs Ham):")
    print(df_emails[label_col].value_counts())

    X = df_emails[feature_cols].values
    y = df_emails[label_col].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # 1. Base counts representation
    X_train_count = X_train
    X_test_count = X_test

    # 2. Normalized term frequency-like representation using MinMaxScaler
    start_time = time.time()
    scaler = MinMaxScaler()
    X_train_norm = scaler.fit_transform(X_train)
    X_test_norm = scaler.transform(X_test)
    normalization_time = time.time() - start_time

    def evaluate_model(model, X_tr, X_te, y_tr, y_te, representation_name):
        t0 = time.time()
        model.fit(X_tr, y_tr)
        train_time = time.time() - t0

        t1 = time.time()
        preds = model.predict(X_te)
        pred_time = time.time() - t1

        print(f"\nModel: {type(model).__name__} ({representation_name})")
        print(f"  Accuracy: {accuracy_score(y_te, preds):.4f}")
        print(f"  Precision: {precision_score(y_te, preds, zero_division=0):.4f}")
        print(f"  Recall: {recall_score(y_te, preds, zero_division=0):.4f}")
        print(f"  F1-Score: {f1_score(y_te, preds, zero_division=0):.4f}")
        print(f"  Training Time: {train_time:.4f}s | Prediction Time: {pred_time:.4f}s")

    # Evaluate on Count counts
    clf_count = MultinomialNB()
    evaluate_model(clf_count, X_train_count, X_test_count, y_train, y_test, "Pre-vectorized Counts")

    # Evaluate on Normalized counts
    clf_norm = LogisticRegression(max_iter=1000)
    evaluate_model(clf_norm, X_train_norm, X_test_norm, y_train, y_test, "Normalized Frequencies")
else:
    print(f"'{email_csv_path}' not found in the current directory.")

# ==========================================
# PART C: Representation Improvement
# ==========================================

print("\n--- Processing Part C: Representation Improvement ---")
print("Improvement strategy: Binarizing counts (word occurrences presence indicator) + Logistic Regression")

if os.path.exists(email_csv_path):
    X_train_bin = (X_train > 0).astype(float)
    X_test_bin = (X_test > 0).astype(float)

    improved_clf = LogisticRegression(max_iter=1000)
    evaluate_model(improved_clf, X_train_bin, X_test_bin, y_train, y_test, "Binarized Occurrences indicator")


--- Processing Part A: Images ---
Extracted image feature shape: (400, 7)

--- Processing Part B: Pre-vectorized Email Data ---
Email dataset shape: (5172, 3002)

Class Distribution (Spam vs Ham):
Prediction
0    3672
1    1500
Name: count, dtype: int64

Model: MultinomialNB (Pre-vectorized Counts)
  Accuracy: 0.9420
  Precision: 0.8681
  Recall: 0.9433
  F1-Score: 0.9042
  Training Time: 0.0788s | Prediction Time: 0.0180s

Model: LogisticRegression (Normalized Frequencies)
  Accuracy: 0.9700
  Precision: 0.9559
  Recall: 0.9400
  F1-Score: 0.9479
  Training Time: 0.5436s | Prediction Time: 0.0067s

--- Processing Part C: Representation Improvement ---
Improvement strategy: Binarizing counts (word occurrences presence indicator) + Logistic Regression

Model: LogisticRegression (Binarized Occurrences indicator)
  Accuracy: 0.9787
  Precision: 0.9542
  Recall: 0.9733
  F1-Score: 0.9637
  Training Time: 1.2214s | Prediction Time: 0.0084s


In [8]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generate_visualizations_and_table(base_dir="448", output_dir="output_visualizations", sample_limit=3):
    """
    Processes images to extract features, saves sample edge-detection visualizations,
    and exports the full feature table to CSV.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    data = []
    sample_count = 0

    print("Processing images and generating feature table...")

    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, file)

                # Determine label (e.g., 1 for Crack, 0 for NonCrack)
                if "NonCrack" in root:
                    label = "Non-Crack"
                    label_id = 0
                else:
                    label = "Crack"
                    label_id = 1

                # Read image
                img = cv2.imread(img_path)
                if img is None:
                    continue

                # 1. Preprocessing: Resize & Grayscale
                target_size = (128, 128)
                img_resized = cv2.resize(img, target_size)
                gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

                # 2. NumPy Intensity Statistics
                mean_brightness = np.mean(gray)
                contrast = np.std(gray)
                dark_pixel_ratio = np.sum(gray < 50) / gray.size
                bright_pixel_ratio = np.sum(gray > 200) / gray.size

                # 3. OpenCV Canny Edge Detection
                edges = cv2.Canny(gray, threshold1=100, threshold2=200)
                edge_count = np.count_nonzero(edges)
                edge_density = edge_count / edges.size

                # Save a few sample visualizations as required by deliverables
                if sample_count < sample_limit:
                    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                    axes[0].imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB))
                    axes[0].set_title(f"Resized Original ({label})")
                    axes[0].axis('off')

                    axes[1].imshow(gray, cmap='gray')
                    axes[1].set_title("Grayscale Representation")
                    axes[1].axis('off')

                    axes[2].imshow(edges, cmap='gray')
                    axes[2].set_title("Canny Edges")
                    axes[2].axis('off')

                    sample_file_path = os.path.join(output_dir, f"sample_{sample_count}_{label}.png")
                    plt.savefig(sample_file_path, bbox_inches='tight')
                    plt.close()
                    print(f"Saved visualization sample to: {sample_file_path}")
                    sample_count += 1

                # Append row to feature list
                data.append({
                    "filename": file,
                    "category": label,
                    "mean_brightness": mean_brightness,
                    "contrast": contrast,
                    "dark_pixel_ratio": dark_pixel_ratio,
                    "bright_pixel_ratio": bright_pixel_ratio,
                    "edge_count": edge_count,
                    "edge_density": edge_density,
                    "label": label_id
                })

    # Create Feature Table DataFrame
    df_features = pd.DataFrame(data)

    # Save to CSV
    csv_output_path = "image_feature_table.csv"
    df_features.to_csv(csv_output_path, index=False)
    print(f"\nFeature table successfully saved to '{csv_output_path}' with {len(df_features)} rows.")

    return df_features

# Run the code
if __name__ == "__main__":
    if os.path.exists("448"):
        feature_df = generate_visualizations_and_table(base_dir="448")
        print("\nFirst 5 rows of the extracted feature table:")
        print(feature_df.head())
    else:
        print("Directory '448' not found. Please extract your dataset archive into the workspace directory.")

Processing images and generating feature table...
Saved visualization sample to: output_visualizations/sample_0_Crack.png
Saved visualization sample to: output_visualizations/sample_1_Crack.png
Saved visualization sample to: output_visualizations/sample_2_Crack.png

Feature table successfully saved to 'image_feature_table.csv' with 400 rows.

First 5 rows of the extracted feature table:
                                     filename category  mean_brightness  \
0                    IMG_8466 resized_448.jpg    Crack       170.720520   
1  IMG_20180514_165217078_HDR resized_448.jpg    Crack       124.947205   
2                    IMG_8476 resized_448.jpg    Crack       176.889771   
3  IMG_20180514_170015428_HDR resized_448.jpg    Crack       140.620789   
4                    IMG_8465 resized_448.jpg    Crack       173.337097   

    contrast  dark_pixel_ratio  bright_pixel_ratio  edge_count  edge_density  \
0  26.622952          0.000000            0.137146        5346      0.326294   